# ComfortPy Simple Tutorial — Core Features

**No optional extras needed — just numpy, pandas, and pythermalcomfort.**

This notebook covers the complete core ComfortPy workflow:

1. **Load** sensor data with `SensorData`
2. **Evaluate** each domain (thermal, visual, acoustic, IAQ)
3. **Combine** into a Global IEQ Index (0–100)
4. **Weighting** presets for different building types
5. **Compliance** and contract-ready reports
6. **Missing domains** — graceful weight renormalization

## 1. Load Sensor Data

`SensorData` wraps a pandas DataFrame and auto-detects column names using built-in aliases (`DEFAULT_ALIASES`). It validates physical bounds and handles NaNs.

In [ ]:
import numpy as np
import pandas as pd
from comfortpy import SensorData

# Simulate 8 hours of sensor readings (1 per hour)
n = 8
rng = np.random.default_rng(seed=42)

df = pd.DataFrame({
    "temperature": 22 + rng.normal(0, 1, n),       # air temp °C
    "radiant_temp": 22 + rng.normal(0, 1, n),       # mean radiant temp °C
    "rh": 50 + rng.normal(0, 5, n),                 # relative humidity %
    "air_speed": 0.1 + rng.normal(0, 0.02, n),      # air velocity m/s
    "lux": 450 + rng.normal(0, 50, n),              # illuminance lux
    "laeq": 45 + rng.normal(0, 5, n),               # noise L_Aeq dB
    "co2": 700 + rng.normal(0, 100, n),             # CO₂ ppm
})

df

In [ ]:
# Wrap with SensorData — auto-detects columns via DEFAULT_ALIASES
sensor = SensorData(df)

print("Detected column mapping:")
for original, canonical in sensor.column_map.items():
    print(f"  '{original}' -> '{canonical}'")

print(f"\nAvailable IEQ domains: {sensor.available_domains()}")

# Validate (checks physical bounds, handles NaNs)
sensor.validate(nan_strategy="interpolate")
print("Validation: OK")

## 2. Evaluate Individual Domains

Each domain function takes NumPy arrays (or scalars) and returns a result dataclass with scores, compliance flags, and metadata.

In [ ]:
from comfortpy import evaluate_thermal, evaluate_visual, evaluate_acoustic, evaluate_iaq

# --- Thermal (ISO 7730 / ASHRAE 55) ---
# Requires: air temp, radiant temp, air velocity, humidity, metabolic rate, clothing
thermal = evaluate_thermal(
    tdb=df["temperature"].values,
    tr=df["radiant_temp"].values,
    vr=df["air_speed"].values,
    rh=df["rh"].values,
    met=1.2,    # sedentary office work
    clo=0.5,    # typical summer clothing
    category="B",  # ISO 7730 category B (normal expectations)
)

print("Thermal (ISO 7730 Category B):")
print(f"  PMV:       {np.round(thermal.pmv, 2)}")
print(f"  PPD:       {np.round(thermal.ppd, 1)}%")
print(f"  Compliant: {thermal.compliant}  (PMV in [-0.5, +0.5])")
print(f"  Category:  {thermal.category}")

In [ ]:
# --- Visual (EN 12464-1) ---
# Requires: illuminance (lux), task type
visual = evaluate_visual(
    illuminance=df["lux"].values,
    task_type="office_writing",  # target: 500 lux
)

print(f"Visual (EN 12464-1, office_writing = {visual.target_lux} lux target):")
print(f"  Illuminance: {np.round(visual.illuminance, 0)} lux")
print(f"  Score:       {np.round(visual.score, 1)}")
print(f"  Compliant:   {visual.compliant}  (illuminance >= target)")

In [ ]:
# --- Acoustic (Noise Criteria) ---
# Requires: L_Aeq (dB), NC level
acoustic = evaluate_acoustic(
    laeq=df["laeq"].values,
    nc_level="NC-35",  # suitable for open-plan offices
)

print(f"Acoustic (NC-35, threshold = {acoustic.threshold_db} dBA):")
print(f"  L_Aeq:      {np.round(acoustic.laeq, 1)} dB")
print(f"  Score:       {np.round(acoustic.score, 1)}")
print(f"  Compliant:   {acoustic.compliant}  (L_Aeq <= threshold)")

# --- IAQ (CO₂-based ventilation adequacy) ---
# Requires: CO₂ (ppm), threshold level
iaq = evaluate_iaq(
    co2=df["co2"].values,
    threshold_level="good",  # 1000 ppm threshold
)

print(f"\nIAQ (threshold 'good' = {iaq.threshold_ppm} ppm):")
print(f"  CO₂:        {np.round(iaq.co2, 0)} ppm")
print(f"  Score:       {np.round(iaq.score, 1)}")
print(f"  Compliant:   {iaq.compliant}  (CO₂ <= threshold)")

## 3. Global IEQ Index

`calculate_global_ieq()` combines all domain results into a single 0–100 index using configurable weights. Missing domains are handled by renormalizing weights.

In [ ]:
from comfortpy import calculate_global_ieq

# Default weights: thermal 40%, IAQ 25%, visual 20%, acoustic 15%
# (Pierson et al. 2019)
global_result = calculate_global_ieq(
    thermal=thermal,
    visual=visual,
    acoustic=acoustic,
    iaq=iaq,
)

print(f"Global IEQ Index: {np.round(global_result.index, 1)}")
print(f"\nPer-domain scores:")
for domain, score in global_result.domain_scores.items():
    print(f"  {domain:12s}: mean={np.mean(score):.1f}, range=[{np.min(score):.1f}, {np.max(score):.1f}]")

print(f"\nWeights used: {global_result.weights_used}")
print(f"Domains:      {global_result.domains}")
print(f"Timestamps:   {global_result.n_timestamps}")

# Interpretation
mean_index = np.mean(global_result.index)
print(f"\nMean IEQ Index: {mean_index:.1f}")
if mean_index >= 80:
    print("  -> Excellent comfort conditions")
elif mean_index >= 60:
    print("  -> Good comfort conditions")
elif mean_index >= 40:
    print("  -> Moderate comfort — some domains need attention")
else:
    print("  -> Poor comfort — multiple domains below standard")

## 4. Weighting Presets

ComfortPy ships with presets for common building types. The same sensor data can yield different IEQ indices depending on which domains matter most for the use case.

In [ ]:
from comfortpy.integration.weights import preset_weights, WEIGHT_PRESETS

print("Available presets:")
for name, weights in WEIGHT_PRESETS.items():
    print(f"  {name:12s}: T={weights['thermal']:.0%}  V={weights['visual']:.0%}"
          f"  A={weights['acoustic']:.0%}  IAQ={weights['iaq']:.0%}")

print("\nComparing presets on the same data:")
print(f"  {'Preset':<12s} {'Mean IEQ':>10s}")
print(f"  {'-' * 24}")

results = {}
for preset_name in ["default", "equal", "office", "school", "healthcare"]:
    weights = preset_weights(preset_name)
    result = calculate_global_ieq(
        thermal=thermal, visual=visual, acoustic=acoustic, iaq=iaq,
        weights=weights,
    )
    results[preset_name] = result
    print(f"  {preset_name:<12s} {np.mean(result.index):>10.1f}")

print("\n'healthcare' weights IAQ highest (40%), 'office' weights thermal highest (45%).")

## 5. Compliance & Performance Contracts

`calculate_compliance()` computes what percentage of timestamps met a threshold IEQ Index. The resulting `ComplianceReport` can be exported as JSON or as a Solidity-ready contract payload.

In [ ]:
from comfortpy import calculate_compliance

report = calculate_compliance(global_result, threshold=60.0)

print(f"Compliance Report:")
print(f"  Threshold:         IEQ Index >= {report.threshold}")
print(f"  Compliance rate:   {report.compliance_rate_pct:.1f}%")
print(f"  IEQ avg:           {report.ieq_index_avg:.1f}")
print(f"  IEQ std:           {report.ieq_index_std:.1f}")
print(f"  IEQ min/max:       {report.ieq_index_min:.1f} / {report.ieq_index_max:.1f}")
print(f"  Compliant hours:   {report.compliant_hours:.1f} / {report.total_hours:.1f}")

print(f"\nPer-domain compliance:")
for domain, rate in report.domain_compliance.items():
    print(f"  {domain:12s}: {rate:.1f}%")

In [ ]:
# Export as JSON (ready for storage or smart contract oracle)
import json

json_str = report.to_json()
print(f"JSON report ({len(json_str)} chars):\n")
print(json.dumps(json.loads(json_str), indent=2)[:500], "...")

In [ ]:
# Export as contract payload (maps to Solidity function parameters)
payload = report.to_contract_payload()
print("Contract payload fields:")
for key, value in payload.items():
    print(f"  {key}: {value}")

## 6. Missing Domain Handling

If you don't have sensors for every domain, ComfortPy renormalizes the weights so the index is still meaningful.

In [ ]:
# Only thermal + IAQ available (no visual or acoustic sensors)
partial = calculate_global_ieq(thermal=thermal, iaq=iaq)

print("Only thermal + IAQ provided:")
print(f"  IEQ Index:  {np.round(partial.index, 1)}")
print(f"  Weights:    {partial.weights_used}")
print(f"  Domains:    {partial.domains}")
print("  -> Weights renormalized so thermal + IAQ sum to 1.0")

# Single domain only
thermal_only = calculate_global_ieq(thermal=thermal)
print(f"\nThermal only:")
print(f"  IEQ Index:  {np.round(thermal_only.index, 1)}")
print(f"  Weights:    {thermal_only.weights_used}")

## Next Steps

- Try the **advanced tutorial**: `examples/advanced_domains_tutorial.ipynb`
- Read the full **User Guide**: `GUIDE.md`
- Install advanced extras: `pip install comfortpy[full]`

# This cell intentionally left blank — delete it.